# Lab 12A — Diagnostico con Spark UI y Plan de Ejecucion

**Sesion 12 | Databricks Data Engineer Associate**  

**Runtime minimo:** DBR 13.3 LTS  
**Archivos fuente:** `ventas_detalle.csv`, `productos_catalogo.csv`

## Objetivos
- Leer y entender el plan de ejecucion con `EXPLAIN FORMATTED`
- Identificar nodos `Exchange` (shuffle) y `BroadcastExchange` en el plan fisico
- Observar en Spark UI la diferencia entre Sort Merge Join y Broadcast Hash Join
- Verificar que AQE esta activo y entender su impacto en particiones post-shuffle

## Setup previo
Subir los archivos al Volume:  
`/Volumes/dbassociate/default/vol_landing/sesion12/`

## withColumn() vs withColumns()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("withColumn vs withColumns").getOrCreate()

# ---------------------------------------------------------
# 1. DataFrame con datos dummy
# ---------------------------------------------------------
data = [
    (1, "Ana", 25, "Lima", 1500.0),
    (2, "Luis", 30, "Arequipa", 2300.0),
    (3, "Marta", 22, "Cusco", 1800.0),
    (4, "Pedro", 40, "Trujillo", 3200.0),
    (5, "Sofia", 35, "Piura", 2750.0),
]

columns = ["id", "nombre", "edad", "ciudad", "salario"]

df = spark.createDataFrame(data, columns)

# ---------------------------------------------------------
# 2. DataFrame usando withColumn 5 veces (encadenado)
# ---------------------------------------------------------
df_chain = (
    df.withColumn("salario_anual", F.col("salario") * 12)
      .withColumn("edad_en_5_anios", F.col("edad") + 5)
      .withColumn("es_mayor_edad", F.col("edad") >= 18)
      .withColumn("nombre_mayus", F.upper(F.col("nombre")))
      .withColumn("bono", F.col("salario") * 0.1)
)

print("=== PLAN LÓGICO/FÍSICO CON withColumn (5 veces) ===")
df_chain.explain(True)

In [0]:
# ---------------------------------------------------------
# 3. DataFrame usando withColumns (una sola llamada, Spark 3.3+)
# ---------------------------------------------------------
df_multi = df.withColumns({
    "salario_anual": F.col("salario") * 12,
    "edad_en_5_anios": F.col("edad") + 5,
    "es_mayor_edad": F.col("edad") >= 18,
    "nombre_mayus": F.upper(F.col("nombre")),
    "bono": F.col("salario") * 0.1,
})

print("=== PLAN LÓGICO/FÍSICO CON withColumns (una sola vez) ===")
df_multi.explain(True)

## Paso 0 — Verificacion del entorno

In [0]:
display(dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/"))

In [0]:
CATALOG       = "dbassociate"
SCHEMA_BRONZE = "bronze"
VOLUME_PATH   = "/Volumes/dbassociate/default/vol_landing/sesion12"

SOURCE_VENTAS    = f"{VOLUME_PATH}/ventas_detalle.csv"
SOURCE_PRODUCTOS = f"{VOLUME_PATH}/productos_catalogo.csv"

print(f"Archivo ventas   : {SOURCE_VENTAS}")
print(f"Archivo productos: {SOURCE_PRODUCTOS}")

## Paso 1 — Leer archivos con esquema explicito

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema_ventas = StructType([
    StructField("venta_id",        StringType(),  False),
    StructField("fecha_venta",     StringType(),  True),
    StructField("cliente_id",      StringType(),  True),
    StructField("producto_id",     StringType(),  True),
    StructField("cantidad",        IntegerType(), True),
    StructField("precio_unitario", DoubleType(),  True),
    StructField("descuento_pct",   DoubleType(),  True),
    StructField("region",          StringType(),  True),
    StructField("canal",           StringType(),  True),
    StructField("estado_pedido",   StringType(),  True),
    StructField("vendedor_id",     StringType(),  True),
])

schema_productos = StructType([
    StructField("producto_id",     StringType(), False),
    StructField("nombre_producto", StringType(), True),
    StructField("categoria",       StringType(), True),
    StructField("subcategoria",    StringType(), True),
    StructField("costo_unitario",  DoubleType(), True),
    StructField("precio_lista",    DoubleType(), True),
    StructField("proveedor",       StringType(), True),
    StructField("activo",          StringType(), True),
])

df_ventas = (
    spark.read
    .option("header", True)
    .schema(schema_ventas)
    .csv(SOURCE_VENTAS)
)

df_productos = (
    spark.read
    .option("header", True)
    .schema(schema_productos)
    .csv(SOURCE_PRODUCTOS)
)

print(f"Ventas cargadas  : {df_ventas.count()} filas")
print(f"Productos cargados: {df_productos.count()} filas")

## Paso 2 — Persistir como tablas Delta para los labs

In [0]:
(
    df_ventas
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.ventas_detalle")
)

(
    df_productos
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.productos_catalogo")
)

print("Tablas Delta creadas en bronze.")
print(f"  - {CATALOG}.{SCHEMA_BRONZE}.ventas_detalle")
print(f"  - {CATALOG}.{SCHEMA_BRONZE}.productos_catalogo")

## Paso 3 — EXPLAIN: leer el plan de un aggregation

El comando `EXPLAIN FORMATTED` muestra el plan fisico real que Spark ejecutara.  
Buscar el nodo `Exchange hashpartitioning` — ese es el shuffle de red.

**Nota**: No aparecerá un nodo Exchange hashpartitioning explícito en el plan físico. Esto se debe se está ejecutando en Photon (cluster serverless), y Photon optimiza la ejecución de manera diferente.

Lo que se verá en el plan es:

PhotonGroupingAgg (paso 2) - este nodo internamente maneja el shuffle necesario para agrupar por region
El shuffle está abstraído dentro de la operación PhotonGroupingAgg
En un plan de Spark tradicional (sin Photon), verías algo como:


```python
HashAggregate (partial)
+- Exchange hashpartitioning(region#xxx, 200)
   +- HashAggregate (final)
```

In [0]:
# Buscar nodo Exchange antes del HashAggregate
spark.sql("""
    EXPLAIN FORMATTED
    SELECT region, SUM(cantidad * precio_unitario) AS total_ventas
    FROM dbassociate.bronze.ventas_detalle
    GROUP BY region
""").show(truncate=False)

In [0]:
# Ejecutar el aggregation y observar el Job en Spark UI
# Spark UI > Jobs > el job reciente > ver el Stage con Exchange
df_por_region = spark.sql("""
    SELECT region,
           COUNT(*)                            AS num_ventas,
           SUM(cantidad * precio_unitario)     AS total_bruto,
           ROUND(AVG(descuento_pct), 2)        AS desc_promedio
    FROM dbassociate.bronze.ventas_detalle
    GROUP BY region
    ORDER BY total_bruto DESC
""")
display(df_por_region)

## Paso 4 — Forzar Sort Merge Join y ver el plan

Al desactivar el broadcast threshold, Spark usara `SortMergeJoin`.  
El plan fisico mostrara **dos nodos Exchange** (shuffle en ambos lados del join).

In [0]:
# Desactivar broadcast automatico para forzar SortMergeJoin
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

spark.sql("""
    EXPLAIN FORMATTED
    SELECT v.venta_id, v.region, v.fecha_venta,
           p.nombre_producto, p.categoria,
           ROUND(v.cantidad * v.precio_unitario, 2) AS monto_bruto
    FROM dbassociate.bronze.ventas_detalle v
    INNER JOIN dbassociate.bronze.productos_catalogo p
        ON v.producto_id = p.producto_id
""").show(truncate=False)

# Punto de observacion:
# Buscar en el plan: SortMergeJoin y dos Exchange hashpartitioning
# Cada Exchange = un shuffle completo de red

In [0]:
# spark.conf.set("spark.sql.adaptive.enable", "false")
# spark.conf.set("spark.sql.shuffle.partitions", "200")

In [0]:
import time

inicio = time.time()
df_smj = spark.sql("""
    SELECT v.venta_id, v.region, v.fecha_venta,
           p.nombre_producto, p.categoria,
           ROUND(v.cantidad * v.precio_unitario, 2) AS monto_bruto
    FROM dbassociate.bronze.ventas_detalle v
    INNER JOIN dbassociate.bronze.productos_catalogo p
        ON v.producto_id = p.producto_id
""")
df_smj.count()
print(f"Sort Merge Join — tiempo: {time.time() - inicio:.2f} s")
display(df_smj.limit(10))

## Paso 5 — Activar Broadcast Hash Join y comparar el plan

Con el threshold restaurado, el catalogo de productos (pequenio) se broadcastea.  
El plan mostrara un unico `BroadcastExchange` — **cero shuffles en el lado grande**.

In [0]:
# Restaurar el threshold de broadcast (10MB por defecto)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

spark.sql("""
    EXPLAIN FORMATTED
    SELECT v.venta_id, v.region, v.fecha_venta,
           p.nombre_producto, p.categoria,
           ROUND(v.cantidad * v.precio_unitario, 2) AS monto_bruto
    FROM dbassociate.bronze.ventas_detalle v
    INNER JOIN dbassociate.bronze.productos_catalogo p
        ON v.producto_id = p.producto_id
""").show(truncate=False)

# Punto de observacion:
# El plan ahora muestra BroadcastHashJoin y BroadcastExchange en el lado de productos
# La tabla ventas NO se shufflea

In [0]:
from pyspark.sql.functions import broadcast

df_ventas_tbl    = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.ventas_detalle")
df_productos_tbl = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.productos_catalogo")

inicio = time.time()
df_bhj = (
    df_ventas_tbl
    .join(broadcast(df_productos_tbl), on="producto_id", how="inner")
    .select(
        "venta_id", "region", "fecha_venta", "nombre_producto", "categoria",
        (df_ventas_tbl.cantidad * df_ventas_tbl.precio_unitario).alias("monto_bruto")
    )
)
df_bhj.count()
print(f"Broadcast Hash Join — tiempo: {time.time() - inicio:.2f} s")
display(df_bhj.limit(10))

### Resumen: Sort Merge Join vs Broadcast Hash Join

| Caracteristica | Sort Merge Join | Broadcast Hash Join |
|---|---|---|
| Shuffles | 2 (ambos lados) | 0 (solo broadcast del lado pequenio) |
| Uso de red | Alto | Bajo |
| Cuando usarlo | Ambas tablas grandes | Un lado menor que autoBroadcastJoinThreshold |
| Hint | `/*+ MERGE(tabla) */` | `broadcast(df)` o `/*+ BROADCAST(tabla) */` |

## Paso 6 — Verificar AQE y sus configuraciones

In [0]:
print("=== Configuracion de AQE ===")
print(f"AQE habilitado              : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Coalescing habilitado       : {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')}")
print(f"Min particion post-coalesce : {spark.conf.get('spark.sql.adaptive.coalescePartitions.minPartitionSize')}")
print(f"Skew join habilitado        : {spark.conf.get('spark.sql.adaptive.skewJoin.enabled')}")
print(f"Shuffle partitions default  : {spark.conf.get('spark.sql.shuffle.partitions')}")

In [0]:
# Aggregation con AQE activo — ver cuantas particiones genera realmente
from pyspark.sql.functions import spark_partition_id

df_resumen = spark.sql("""
    SELECT
        v.region,
        p.categoria,
        DATE_TRUNC('month', CAST(v.fecha_venta AS DATE)) AS mes,
        COUNT(*)                                          AS num_ventas,
        ROUND(SUM(v.cantidad * v.precio_unitario), 2)     AS total_bruto,
        ROUND(SUM(v.cantidad * v.precio_unitario * (1 - v.descuento_pct / 100)), 2) AS total_neto
    FROM dbassociate.bronze.ventas_detalle v
    JOIN dbassociate.bronze.productos_catalogo p
        ON v.producto_id = p.producto_id
    GROUP BY v.region, p.categoria, DATE_TRUNC('month', CAST(v.fecha_venta AS DATE))
    ORDER BY mes, v.region, p.categoria
""")

display(df_resumen)
num_partitions = df_resumen.select(spark_partition_id().alias("pid")).distinct().count()
print(f"\nParticiones del resultado (post-AQE coalescing): {num_partitions}")

## Paso 7 — Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.ventas_detalle")
spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.productos_catalogo")
print("Tablas eliminadas. Lab 12A completado.")

## Puntos clave del examen

1. `EXPLAIN FORMATTED` muestra el plan fisico — `Exchange hashpartitioning` = shuffle de red
2. `BroadcastHashJoin` no genera shuffle en el lado grande — es el join mas eficiente para tablas pequenias
3. `spark.sql.autoBroadcastJoinThreshold` controla el tamano maximo para broadcast automatico (default: 10MB)
4. AQE esta habilitado por defecto en DBR 8.0+ — coalesce particiones post-shuffle automaticamente
5. `spark.sql.shuffle.partitions` controla el numero de reducers en aggregations y joins (default: 200)